# 04 · 실험에서 기획서 초안으로

2Pager(2페이지 제안 요약)·PRD(Product Requirements Document, 제품 요구사항 문서)의 핵심을 실행 근거와 연결합니다. HTML은 문서 시안이며 서비스 구현이 아닙니다. 약 15분.

모델 다운로드·API 키·GPU 없이 실행합니다. 합성 자료만 사용하며, 실제 LLM 학습·성능 검증과 구분합니다. 예측 → 실행 → 변수 변경 → 근거 기록의 순서로 진행합니다.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
from IPython.display import display, HTML
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course_labs').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from course_labs.adaptation import *
OUT = ROOT / 'output' / 'validation' / 'adaptation-labs'
OUT.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0], 'NumPy:', np.__version__)
assert sys.version_info[:2] == (3, 11), '공통 환경 Python 3.11 커널을 선택하세요.'


## 서비스 문제의 범위

글쓰기 개선 서비스의 합성 기획 예시입니다. 아래 내용을 복사 제출하지 말고 자신의 문제로 바꾸세요.

In [ ]:
doc = {'user':'한국어 논증 글쓰기를 연습하는 대학생', 'problem':'자신의 주장과 근거가 연결되는지 확인하기 어려움. 문단별 근거 피드백 제안.', 'baseline':'교수·동료 피드백 또는 일반 채팅 도구', 'scope':'서론·본론1·본론2·결론 구조와 근거 연결만 지원. 성적 자동 결정 제외.', 'data_rights':'동의받은 글 또는 합성 글. 이름 제거·삭제 요청 경로 제공.', 'evaluation':'개발용 20건과 보류용 10건 분리; 구조 판정 일치도·근거 인용 오류·수정 수용 여부 기록.', 'fallback':'근거 부족 시 판단 보류·사용자 수정·교수 검토', 'roadmap':'7주 문제·초안; 8주 피드백 반영; 9–12주 최소 기능·평가; 13–16주 운영·발표', 'question':'구조 판정과 문체 피드백 중 먼저 검증할 범위는 무엇인가?'}
display(doc)

## 기획서 필수 항목

문자열 유무 검사입니다. 좋은 기획서인지 자동 채점하지 않습니다.

In [ ]:
assert proposal_check(doc) == []
print('항목 유무 통과 · 내용 검토 별도')

## 누락의 주입

evaluation이 없어도 시안은 예뻐 보일 수 있습니다. 자동 검사로 누락만 찾습니다.

In [ ]:
incomplete = dict(doc); incomplete.pop('evaluation')
assert proposal_check(incomplete) == ['evaluation']
print(proposal_check(incomplete))

## 기술 대안 비교

서비스 요구를 먼저 정한 뒤 어떤 실패를 고칠지 비교합니다.

In [ ]:
alternatives = [{'method':'Prompt + schema','first_test':'구조 판정과 JSON 오류','cost':'개발시간·추론비용'}, {'method':'RAG','first_test':'평가기준 문서의 근거 회수','cost':'검색·문서 관리'}, {'method':'SFT/LoRA','first_test':'반복 형식·행동 오류','cost':'허가된 학습자료·GPU·재평가'}, {'method':'DPO','first_test':'기준 있는 선호쌍과 편향','cost':'선호 라벨·reference·재평가'}]
display(alternatives)

## 측정 가능한 요구사항

아래 목표는 제안값입니다. 교수 합의나 실제 성과가 아닙니다.

In [ ]:
requirements = [{'id':'FR01','requirement':'주장·근거 문장을 원문 위치로 제시','acceptance':'고정 10건에서 인용 위치 오류 0건 목표'}, {'id':'FR02','requirement':'불확실한 구조는 보류','acceptance':'보류 사유·수정 버튼을 모든 보류 응답에 표시'}]
display(requirements)

## 위험과 복구

사업·사용 경험·개발 관점에서 실패를 점검합니다.

In [ ]:
risks = [{'risk':'학생 글 외부 노출','control':'동의·최소 수집·비공개 저장·삭제'}, {'risk':'유창한 오판','control':'원문 근거·보류·사람 검토'}, {'risk':'구현 범위 과다','control':'구조 판정 한 기능으로 시작'}]
display(risks)

## 실험 근거 연결

앞의 노트북 결과가 있는지 확인합니다. 미실행 결과를 생성한 것처럼 표시하지 않습니다.

In [ ]:
evidence=[]
for name in ['dataset-card.json','low-rank-report.json','preference-report.json']:
    evidence.append({'file':name,'exists':(OUT/name).exists()})
display(evidence)

## 역할과 완료 조건

2–4인 팀은 역할을 나누고, 개인은 같은 항목을 혼자 점검합니다. 학생 이름을 넣지 않습니다.

In [ ]:
tasks = [{'role':'PM','done':'문제·대안·범위·근거 정리'}, {'role':'UX','done':'정상·오류·보류·수정 흐름'}, {'role':'개발·평가','done':'데이터 권한·고정 평가셋·복구 계획'}]
display(tasks)

## HTML 문서 생성

입력 문자열은 escape 처리합니다. 생성 후 브라우저에서 문서를 검토하세요.

In [ ]:
html = proposal_html(doc)
path = OUT/'proposal-draft.html'
path.write_text(html,encoding='utf-8')
print(path)

## 시안 미리보기

이 미리보기는 문서입니다. 실제 서비스나 사용성 검증이 아닙니다.

In [ ]:
display(HTML(html))

## 피드백 기록

피드백을 받은 뒤 채택·보류·반박의 이유를 남깁니다. 아래는 빈 기록 양식입니다.

In [ ]:
feedback = {'version':'0.1','question':doc['question'],'professor_feedback':'미수신','decision':'미결정','reason':'','next_evidence':''}
save_report(OUT/'feedback-log.json',feedback)
display(feedback)

## 제출 전 검사

초안10/16 → 피드백 → 수정본10/23. 소속 공지의 정확한 시각·채널을 확인합니다.

In [ ]:
save_report(OUT/'proposal-draft.json',doc)
print('아주대: LMS / 다른 학교: 공지 교수 이메일')
print('PPT·Word·PDF·HTML 가능 / HWP 제외 / 개인정보 공개 GitHub 금지')

## 실행 기록과 기획서 연결

1. 바꾼 변수와 달라진 결과
2. 이 실험으로 확인하지 못한 부분
3. 프로젝트에서 추가로 검증할 가정

출력은 `output/validation/adaptation-labs/`에 저장됩니다. 개인정보나 학생 제출물은 공개 저장소에 넣지 않습니다.